In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "cellpose",
#     "tqdm"
# ]
# ///

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
from cellpose import core, io, models, plot
from tqdm import tqdm

In [3]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

creating new log file
2026-06-03 20:38:52,158 [INFO] WRITING LOG OUTPUT TO /home/runner/.cellpose/run.log


2026-06-03 20:38:52,159 [INFO] 
cellpose version: 	4.1.1 
platform:       	linux 
python version: 	3.12.3 
torch version:  	2.12.0+cu130


2026-06-03 20:38:52,160 [INFO] Neither TORCH CUDA nor MPS version not installed/working.


GPU available: False


In [4]:
image_path = "../../../_static/images/cellpose/cell_cellpose.tif"
image = io.imread(image_path)  # or image = tifffile.imread(image_path)

print(image.shape)

(2, 1040, 1392)


In [ ]:
model = models.CellposeModel(pretrained_model="cpsam", gpu=use_gpu)

In [ ]:
masks, flows, styles = model.eval(image)

In [ ]:
fig = plt.figure(figsize=(12, 5))
plot.show_segmentation(fig, image, masks, flows[0])
plt.tight_layout()
# Optional if you want to also save the figure
# plt.savefig(f"path/to/output/{Path(image_path).stem}_cp_output.png")
plt.show()

In [ ]:
output_path = f"path/to/output/{Path(image_path).stem}_labels.tif"
io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

In [ ]:
# Path to the folder containing the images to segment
folder_path = Path("data/05_segmentation_cellpose")

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Run Cellpose on each image one by one
# NOTE: tqdm is used to show a progress bar, but you can remove it if you don't want it
for image_path in tqdm(images_path, desc="Processing images"):
    # Load the image
    image = io.imread(image_path)
    # Run Cellpose on the image
    masks, flows, styles = model.eval(image)
    # Save the segmentation results as a TIFF file
    output_path = folder_path / f"{image_path.stem}_labels.tif"
    io.imsave(output_path, masks)  # or tifffile.imwrite(output_path, masks)

In [ ]:
# Path to the folder containing the images to segment
folder_path = Path("data/05_segmentation_cellpose")

# Get the sorted list of all .tif images in the folder
images_path = sorted(folder_path.glob("*.tif"))

# Increase batch_size to reduce GPU passes per image (uses more GPU memory)
batch_size = 8  # each pass sends batch_size tiles of 256×256 to the GPU
for image_path in tqdm(images_path, desc="Processing images"):
    image = io.imread(image_path)
    masks, flows, styles = model.eval(image, batch_size=batch_size)
    output_path = folder_path / f"{image_path.stem}_labels.tif"
    io.imsave(output_path, masks)